# Final 2 experiments

This Colab finishes the two experiments that were still missing from the PI review:

1. **Cross-dataset duplicate and similarity audit.** For every directed source-to-target pair, it compares the source train and validation records with the target test records. It reports exact and near-identical target records, writes a 20-row removal table, and saves a filtered target manifest for every pair.
2. **A second foundation model without benchmark pretraining overlap.** It fine-tunes ECGFounder on each of the five sources and evaluates the complete 5 by 5 matrix. The ECGFounder paper reports pretraining on the Harvard-Emory ECG Database, not PTB-XL, CPSC2018, Georgia, MIMIC-IV-ECG, or CODE-15%.

The notebook is restart-safe. Completed fingerprint caches, source-model checkpoints, and complete result tables are reused from Drive. Use a GPU runtime for the ECGFounder portion. Run the notebook from the top.

In [ ]:
#@title 1. Controls
MODE = "full"  #@param ["smoke", "full"]
RUN_DUPLICATE_AUDIT = True  #@param {type:"boolean"}
RUN_ECGFOUNDER_TRAINING = True  #@param {type:"boolean"}
RUN_ECGFOUNDER_MATRIX = True  #@param {type:"boolean"}
PROJECT_ROOT_OVERRIDE = ""  #@param {type:"string"}
REQUIRE_ALL_FIVE_DATASETS = True
SEED = 42
DATASETS = ["ptbxl", "cpsc2018", "georgia", "mimic_iv", "code_ii"]
if MODE not in {"smoke", "full"}:
    raise ValueError("MODE must be smoke or full")
print({"mode": MODE, "datasets": DATASETS})

In [ ]:
#@title 2. Mount Drive and install the benchmark code
from google.colab import drive, userdata
drive.mount("/content/drive")

import base64
import datetime as dt
import json
import os
import shutil
import subprocess
import sys
import tarfile
import zipfile
from pathlib import Path

REPOSITORY_URL = "https://github.com/tanushappapogu-max/ecg-generalization-benchmark.git"
WORKSPACE = Path("/content/ecg-generalization-benchmark")
try:
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = None
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
if github_token:
    encoded = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
    git_env.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {encoded}",
    })
if WORKSPACE.exists() and not (WORKSPACE / ".git").is_dir():
    raise RuntimeError(f"Partial checkout at {WORKSPACE}; restart the runtime")
if not WORKSPACE.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY_URL, str(WORKSPACE)], env=git_env)
else:
    subprocess.check_call(["git", "-C", str(WORKSPACE), "pull", "--ff-only"], env=git_env)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", f"{WORKSPACE}[train,test]", "faiss-cpu", "huggingface_hub", "seaborn"])
required = [
    "src/evaluation/waveform_overlap_audit.py",
    "src/training/ecg_founder_pipeline.py",
    "src/evaluation/ecg_founder_matrix.py",
]
missing = [name for name in required if not (WORKSPACE / name).is_file()]
if missing:
    raise RuntimeError("The final-two-experiments commit is not on GitHub main yet: " + ", ".join(missing))
sys.path.insert(0, str(WORKSPACE))
commit = subprocess.check_output(["git", "-C", str(WORKSPACE), "rev-parse", "HEAD"], text=True).strip()
print({"repository_commit": commit})

In [ ]:
#@title 3. Locate the shared project, manifests, and waveform files
PROJECT_FOLDER_NAME = "LSTS ECG Generalization Benchmark — START HERE"
def locate_project_root():
    if PROJECT_ROOT_OVERRIDE:
        path = Path(PROJECT_ROOT_OVERRIDE)
        if not path.is_dir():
            raise FileNotFoundError(path)
        return path
    direct = [Path("/content/drive/MyDrive") / PROJECT_FOLDER_NAME, Path("/content/drive/Shareddrives") / PROJECT_FOLDER_NAME]
    for path in direct:
        if path.is_dir():
            return path
    for root in [Path("/content/drive/Shareddrives"), Path("/content/drive/MyDrive")]:
        if not root.exists():
            continue
        for base, directories, _ in os.walk(root):
            if PROJECT_FOLDER_NAME in directories:
                return Path(base) / PROJECT_FOLDER_NAME
    raise FileNotFoundError(f"Cannot locate {PROJECT_FOLDER_NAME!r}; set PROJECT_ROOT_OVERRIDE")

PROJECT_ROOT = locate_project_root()
DATASETS_ROOT = PROJECT_ROOT / "01_DATASETS"
CODE_ROOT = PROJECT_ROOT / "02_CODE_AND_NOTEBOOKS"
RESULTS_ROOT = PROJECT_ROOT / "03_RESULTS_AND_MODEL_OUTPUTS"
prefixes = {
    "ptbxl": "01_PTB_XL", "cpsc2018": "02_CPSC_2018",
    "georgia": "03_GEORGIA_12_LEAD", "mimic_iv": "04_MIMIC_IV_ECG",
    "code_ii": "05_CODE_15_PERCENT",
}
def dataset_folder(name):
    matches = sorted(path for path in DATASETS_ROOT.glob(prefixes[name] + "*") if path.is_dir())
    return matches[0] if matches else DATASETS_ROOT / prefixes[name]
DATASET_FOLDERS = {name: dataset_folder(name) for name in DATASETS}
MANIFEST_ROOT = Path("/content/final-two-manifests")
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)
drive_manifest_root = CODE_ROOT / "03_ECG_FM_TRAINING_AND_EVALUATION" / "manifests"
for name in DATASETS:
    filename = f"{name}_week2.csv"
    candidates = [drive_manifest_root / filename, DATASET_FOLDERS[name] / filename, WORKSPACE / "data" / "week2" / filename]
    source = next((path for path in candidates if path.is_file()), None)
    if source:
        shutil.copy2(source, MANIFEST_ROOT / filename)
missing_manifests = [name for name in DATASETS if not (MANIFEST_ROOT / f"{name}_week2.csv").is_file()]
if missing_manifests:
    raise RuntimeError("Missing frozen manifests: " + ", ".join(missing_manifests))

LOCAL_DATA_ROOT = Path("/content/ecg-final-two-data")
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
def direct_npy_root(name):
    folder = DATASET_FOLDERS[name]
    return folder if (folder / "signals").is_dir() else None

def prepare_georgia():
    direct = direct_npy_root("georgia")
    if direct:
        return direct
    source = DATASET_FOLDERS["georgia"]
    destination = LOCAL_DATA_ROOT / "georgia"
    if (destination / "signals").is_dir():
        return destination
    parts = sorted(source.glob("georgia_signals.tar.gz.part-*"))
    if not parts:
        return source
    archive = Path("/content/georgia_signals.tar.gz")
    expected = sum(path.stat().st_size for path in parts)
    if not archive.is_file() or archive.stat().st_size != expected:
        with archive.open("wb") as output:
            for index, part in enumerate(parts, 1):
                print(f"Georgia archive part {index}/{len(parts)}", flush=True)
                with part.open("rb") as handle:
                    shutil.copyfileobj(handle, output, length=16 * 1024 * 1024)
    destination.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, "r:gz") as handle:
        handle.extractall(destination, filter="data")
    return destination

def prepare_mimic():
    source = DATASET_FOLDERS["mimic_iv"]
    for candidate in [source / "mimic_50k_waveforms", source]:
        if (candidate / "files").is_dir():
            return candidate
    destination = LOCAL_DATA_ROOT / "mimic_iv"
    for candidate in [destination / "mimic_50k_waveforms", destination]:
        if (candidate / "files").is_dir():
            return candidate
    shard_root = source / "mimic_50k_waveforms"
    shards = sorted(shard_root.glob("mimic_50k_waveforms_*-of-050.tar.gz"))
    if len(shards) != 50:
        return source
    destination.mkdir(parents=True, exist_ok=True)
    markers = destination / ".completed_shards"
    markers.mkdir(exist_ok=True)
    for index, archive in enumerate(shards, 1):
        marker = markers / (archive.name + ".done")
        if marker.is_file():
            continue
        print(f"MIMIC shard {index}/50", flush=True)
        with tarfile.open(archive, "r:gz") as handle:
            handle.extractall(destination, filter="data")
        marker.write_text(dt.datetime.now(dt.timezone.utc).isoformat())
    for candidate in [destination / "mimic_50k_waveforms", destination]:
        if (candidate / "files").is_dir():
            return candidate
    return destination

def prepare_code():
    source = DATASET_FOLDERS["code_ii"]
    for candidate in [source, source / "signals"]:
        if all((candidate / f"exams_part{i}.hdf5").is_file() for i in range(3)):
            return candidate
    destination = LOCAL_DATA_ROOT / "code_ii"
    destination.mkdir(parents=True, exist_ok=True)
    for index in range(3):
        target = destination / f"exams_part{index}.hdf5"
        if target.is_file():
            continue
        archive = source / f"exams_part{index}.zip"
        if not archive.is_file():
            raise FileNotFoundError(f"Missing CODE archive {archive}")
        print(f"Extracting {archive.name}", flush=True)
        with zipfile.ZipFile(archive) as handle:
            members = [item for item in handle.infolist() if Path(item.filename).name == target.name]
            if len(members) != 1:
                raise RuntimeError(f"{archive} does not contain exactly one {target.name}")
            with handle.open(members[0]) as input_stream, target.open("wb") as output_stream:
                shutil.copyfileobj(input_stream, output_stream, length=16 * 1024 * 1024)
    return destination

SIGNAL_ROOTS = {
    "ptbxl": direct_npy_root("ptbxl") or DATASET_FOLDERS["ptbxl"],
    "cpsc2018": direct_npy_root("cpsc2018") or DATASET_FOLDERS["cpsc2018"],
    "georgia": prepare_georgia(),
    "mimic_iv": prepare_mimic(),
    "code_ii": prepare_code(),
}
print({"project_root": str(PROJECT_ROOT), "signal_roots": {k: str(v) for k, v in SIGNAL_ROOTS.items()}})

In [ ]:
#@title 4. Preflight all five frozen datasets
import h5py
import numpy as np
import pandas as pd
from src.data.week2_manifest import validate_canonical_manifest

def waveform_exists(root, row):
    storage = str(row["storage"])
    if storage == "hdf5":
        filename, separator, _ = str(row["signal_path"]).rpartition("::")
        return bool(separator) and (root / filename).is_file()
    path = root / str(row["signal_path"])
    if storage == "npy":
        return path.is_file()
    if storage == "wfdb":
        base = path.with_suffix("") if path.suffix in {".hea", ".dat"} else path
        return base.with_suffix(".hea").is_file() and base.with_suffix(".dat").is_file()
    return False

readiness = []
for name in DATASETS:
    try:
        manifest = validate_canonical_manifest(pd.read_csv(MANIFEST_ROOT / f"{name}_week2.csv", low_memory=False))
        checked = manifest if MODE == "full" else manifest.groupby("split", sort=False, group_keys=False).head(64)
        missing = [str(row["signal_path"]) for _, row in checked.iterrows() if not waveform_exists(SIGNAL_ROOTS[name], row)]
        if missing:
            raise FileNotFoundError(f"missing waveform examples: {missing[:5]}")
        readiness.append({"dataset": name, "ready": True, "records": len(manifest), "checked": len(checked), "detail": "PASS"})
    except Exception as exc:
        readiness.append({"dataset": name, "ready": False, "records": 0, "checked": 0, "detail": str(exc)})
readiness = pd.DataFrame(readiness)
display(readiness)
READY_DATASETS = readiness.loc[readiness["ready"], "dataset"].tolist()
if REQUIRE_ALL_FIVE_DATASETS and set(READY_DATASETS) != set(DATASETS):
    raise RuntimeError("All five datasets are required; fix the failed preflight rows before running experiments")
if len(READY_DATASETS) < 2:
    raise RuntimeError("At least two datasets must pass preflight")
FINAL_ROOT = RESULTS_ROOT / "FINAL_2_EXPERIMENTS" / MODE
FINAL_ROOT.mkdir(parents=True, exist_ok=True)
print({"ready_datasets": READY_DATASETS, "output_root": str(FINAL_ROOT)})

In [ ]:
#@title 5. Run the 20-direction duplicate and near-similarity audit
DUPLICATE_ROOT = FINAL_ROOT / "01_CROSS_DATASET_DUPLICATE_AUDIT"
duplicate_summary_path = DUPLICATE_ROOT / "source_target_removal_summary.csv"
duplicate_already_complete = False
if duplicate_summary_path.is_file():
    previous = pd.read_csv(duplicate_summary_path)
    duplicate_already_complete = len(previous) == len(READY_DATASETS) * (len(READY_DATASETS) - 1)
if RUN_DUPLICATE_AUDIT and duplicate_already_complete:
    print("Reusing the completed directional duplicate audit")
elif RUN_DUPLICATE_AUDIT:
    command = [
        sys.executable, "-m", "src.evaluation.waveform_overlap_audit",
        "--output-dir", str(DUPLICATE_ROOT),
        "--datasets", *READY_DATASETS,
        "--candidate-cosine", "0.97",
        "--median-correlation", "0.995",
        "--median-nrmse", "0.15",
        "--max-lag-ms", "100",
        "--top-k", "5",
    ]
    for name in READY_DATASETS:
        command += ["--manifest", f"{name}={MANIFEST_ROOT / f'{name}_week2.csv'}"]
        command += ["--signal-root", f"{name}={SIGNAL_ROOTS[name]}"]
    if MODE == "smoke":
        command += ["--max-records-per-partition", "64", "--progress-every", "16"]
    subprocess.check_call(command, cwd=WORKSPACE, env={**os.environ, "PYTHONUNBUFFERED": "1"})
else:
    print("Duplicate audit disabled")
if duplicate_summary_path.is_file():
    duplicate_summary = pd.read_csv(duplicate_summary_path)
    display(duplicate_summary)
    print("Pairwise removal table:", duplicate_summary_path)
    print("Record-level matches:", DUPLICATE_ROOT / "all_cross_dataset_matches.csv")
    print("Filtered manifests:", DUPLICATE_ROOT / "deduplicated_manifests")

## Why ECGFounder is the second foundation model

The published ECGFounder model was pretrained on 10,771,552 ECGs from the Harvard-Emory ECG Database. The paper treats CODE and PTB-XL as external validation sets and MIMIC-IV-ECG as downstream fine-tuning data. The five benchmark datasets are therefore not reported as part of the checkpoint's pretraining set. This directly addresses the overlap concern raised for ECG-FM.

The next cell pins the official implementation to a specific commit and downloads the public 12-lead checkpoint from the authors' Hugging Face repository.

In [ ]:
#@title 6. Fetch the pinned official ECGFounder code and checkpoint
ECGFOUNDER_REPOSITORY = Path("/content/ECGFounder")
ECGFOUNDER_COMMIT = "04edac702b61c91face519774ddcc0cd712fef23"
if not ECGFOUNDER_REPOSITORY.exists():
    subprocess.check_call(["git", "clone", "https://github.com/PKUDigitalHealth/ECGFounder.git", str(ECGFOUNDER_REPOSITORY)])
subprocess.check_call(["git", "-C", str(ECGFOUNDER_REPOSITORY), "fetch", "origin", ECGFOUNDER_COMMIT])
subprocess.check_call(["git", "-C", str(ECGFOUNDER_REPOSITORY), "checkout", "--detach", ECGFOUNDER_COMMIT])
from huggingface_hub import hf_hub_download
ECGFOUNDER_CHECKPOINT = Path(hf_hub_download(
    repo_id="PKUDigitalHealth/ECGFounder",
    filename="12_lead_ECGFounder.pth",
    local_dir="/content/checkpoints/ecgfounder",
))
if ECGFOUNDER_CHECKPOINT.stat().st_size < 300_000_000:
    raise RuntimeError("The ECGFounder checkpoint is unexpectedly small")
print({"official_commit": ECGFOUNDER_COMMIT, "checkpoint": str(ECGFOUNDER_CHECKPOINT), "bytes": ECGFOUNDER_CHECKPOINT.stat().st_size})

In [ ]:
#@title 7. Run focused implementation tests before spending GPU time
subprocess.check_call([sys.executable, "-m", "pytest", "-q", "tests/test_final_two_experiments.py"], cwd=WORKSPACE)
print("Final-two-experiments tests: PASS")

In [ ]:
#@title 8. Fine-tune ECGFounder on all five source datasets
if RUN_ECGFOUNDER_TRAINING and not __import__("torch").cuda.is_available():
    raise RuntimeError("ECGFounder training requires a GPU runtime")
ECGFOUNDER_RUNS = FINAL_ROOT / "02_ECGFOUNDER_SOURCE_RUNS"
PREPROCESSING_CACHE = Path("/content/ecgfounder-preprocessed-cache")
def completed_training(output_dir):
    metrics = output_dir / "test_metrics.json"
    checkpoint = output_dir / "best_checkpoint.pt"
    if not metrics.is_file() or not checkpoint.is_file():
        return False
    expected = "SMOKE_PASS" if MODE == "smoke" else "COMPLETE"
    return json.loads(metrics.read_text()).get("status") == expected

if RUN_ECGFOUNDER_TRAINING:
    for name in READY_DATASETS:
        output_dir = ECGFOUNDER_RUNS / name
        if completed_training(output_dir):
            print(f"Reusing completed ECGFounder source run: {name}")
            continue
        command = [
            sys.executable, "-m", "src.training.ecg_founder_pipeline",
            "--dataset", name,
            "--manifest", str(MANIFEST_ROOT / f"{name}_week2.csv"),
            "--signal-root", str(SIGNAL_ROOTS[name]),
            "--ecg-founder-repository", str(ECGFOUNDER_REPOSITORY),
            "--pretrained-checkpoint", str(ECGFOUNDER_CHECKPOINT),
            "--output-dir", str(output_dir),
            "--preprocessing-cache", str(PREPROCESSING_CACHE),
            "--seed", str(SEED), "--resume",
        ]
        if MODE == "smoke":
            command += ["--smoke-test", "--epochs", "1", "--patience", "1", "--max-records-per-split", "64", "--batch-size", "2", "--gradient-accumulation-steps", "1", "--no-mixed-precision"]
        else:
            command += ["--epochs", "5", "--patience", "3", "--batch-size", "8", "--gradient-accumulation-steps", "4", "--learning-rate", "1e-4"]
        subprocess.check_call(command, cwd=WORKSPACE, env={**os.environ, "PYTHONUNBUFFERED": "1"})
else:
    print("ECGFounder training disabled")

In [ ]:
#@title 9. Evaluate the complete ECGFounder source-to-target matrix
ECGFOUNDER_MATRIX = FINAL_ROOT / "03_ECGFOUNDER_5_BY_5_MATRIX"
matrix_path = ECGFOUNDER_MATRIX / "ecg_founder_five_label_matrix_long.csv"
matrix_summary_path = ECGFOUNDER_MATRIX / "matrix_summary.json"
matrix_already_complete = False
if matrix_summary_path.is_file():
    previous_matrix_summary = json.loads(matrix_summary_path.read_text())
    matrix_already_complete = previous_matrix_summary.get("completed_cells") == len(READY_DATASETS) ** 2 and previous_matrix_summary.get("blocked_cells") == 0
if RUN_ECGFOUNDER_MATRIX and matrix_already_complete:
    print("Reusing the completed ECGFounder matrix")
elif RUN_ECGFOUNDER_MATRIX:
    if not all(completed_training(ECGFOUNDER_RUNS / name) for name in READY_DATASETS):
        raise RuntimeError("Every ECGFounder source run must finish before the matrix")
    command = [
        sys.executable, "-m", "src.evaluation.ecg_founder_matrix",
        "--source-runs-root", str(ECGFOUNDER_RUNS),
        "--manifest-root", str(MANIFEST_ROOT),
        "--ecg-founder-repository", str(ECGFOUNDER_REPOSITORY),
        "--pretrained-checkpoint", str(ECGFOUNDER_CHECKPOINT),
        "--output-dir", str(ECGFOUNDER_MATRIX),
        "--preprocessing-cache", str(PREPROCESSING_CACHE),
        "--datasets", *READY_DATASETS,
        "--seed", str(SEED), "--fail-on-missing",
    ]
    for name in READY_DATASETS:
        command += ["--signal-root", f"{name}={SIGNAL_ROOTS[name]}"]
    if MODE == "smoke":
        command += ["--max-records-per-target", "64", "--batch-size", "2", "--no-mixed-precision"]
    subprocess.check_call(command, cwd=WORKSPACE, env={**os.environ, "PYTHONUNBUFFERED": "1"})
else:
    print("ECGFounder matrix disabled")
if matrix_path.is_file():
    matrix_long = pd.read_csv(matrix_path)
    display(matrix_long)
    display(matrix_long.pivot(index="source_dataset", columns="target_dataset", values="macro_auroc"))

In [ ]:
#@title 10. Final completion audit and files to send the team
audit_rows = []
duplicate_summary_path = DUPLICATE_ROOT / "source_target_removal_summary.csv"
if duplicate_summary_path.is_file():
    duplicate_summary = pd.read_csv(duplicate_summary_path)
    audit_rows.append({
        "experiment": "20-direction duplicate audit",
        "complete": len(duplicate_summary) == len(READY_DATASETS) * (len(READY_DATASETS) - 1),
        "deliverable": str(duplicate_summary_path),
    })
else:
    audit_rows.append({"experiment": "20-direction duplicate audit", "complete": False, "deliverable": str(duplicate_summary_path)})
training_complete = all(completed_training(ECGFOUNDER_RUNS / name) for name in READY_DATASETS)
audit_rows.append({"experiment": "ECGFounder five-source fine-tuning", "complete": training_complete, "deliverable": str(ECGFOUNDER_RUNS)})
if matrix_path.is_file():
    matrix_long = pd.read_csv(matrix_path)
    matrix_complete = len(matrix_long) == len(READY_DATASETS) ** 2 and matrix_long["status"].eq("COMPLETE").all()
else:
    matrix_complete = False
audit_rows.append({"experiment": "ECGFounder complete source-target matrix", "complete": matrix_complete, "deliverable": str(matrix_path)})
audit = pd.DataFrame(audit_rows)
display(audit)
audit.to_csv(FINAL_ROOT / "FINAL_2_EXPERIMENTS_COMPLETION_AUDIT.csv", index=False)
if audit["complete"].all():
    print("FINAL_TWO_EXPERIMENTS_COMPLETE")
    print("Duplicate counts:", duplicate_summary_path)
    print("All matched records:", DUPLICATE_ROOT / "all_cross_dataset_matches.csv")
    print("Deduplicated manifests:", DUPLICATE_ROOT / "deduplicated_manifests")
    print("ECGFounder long-form results:", matrix_path)
    print("ECGFounder AUROC matrix:", ECGFOUNDER_MATRIX / "ecg_founder_five_label_macro_auroc_matrix.csv")
else:
    print("NOT COMPLETE. Rerun from the top; completed stages will be reused.")